# 🎬 Movie Review Sentiment Analysis: End-to-End NLP Pipeline
**Applied Machine Learning Portfolio — Natural Language Processing**


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-username/movie-review-sentiment-analysis/blob/main/notebooks/Movie_Review_Sentiment_Analysis.ipynb)

---

### Project Objectives & Technical Scope
1. **Binary Sentiment Classification**: Predict whether an unstructured movie review is **Positive** or **Negative**.
2. **Reproducible Dataset Acquisition**: Benchmark using the Stanford Large Movie Review Dataset (IMDb 50,000 reviews by Andrew Maas et al.).
3. **Data Leakage Prevention**: Enforce strict separation using stratified **Train (70%) / Validation (15%) / Test (15%)** partitions and fitting feature extractors exclusively inside scikit-learn Pipelines.
4. **Model Comparison**: Benchmark **Multinomial Naive Bayes** against **Logistic Regression** with sublinear TF-IDF n-grams (1-2).
5. **Comprehensive Evaluation**: Assess accuracy, precision, recall, F1-score, confusion matrix, ROC-AUC, and qualitative error analysis.
6. **Production Export**: Serialize the champion pipeline with `joblib` for deployment in our interactive Streamlit application.


In [ ]:
# 1. Environment Setup and Reproducibility
import os
import sys
import time
import html
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve
)

# Set global random seed for complete reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print("Environment initialized successfully with fixed random seed.")


---
## 2. Dataset Acquisition & Provenance
We utilize the **Stanford Large Movie Review Dataset** (IMDb 50k), originally collected by Andrew L. Maas et al. (ACL 2011).
- **Size**: 50,000 highly polar movie reviews.
- **Balance**: 25,000 positive reviews, 25,000 negative reviews.
- **License / Terms**: Freely accessible for academic and research benchmarking.

The cell below ensures automated, reproducible download directly into your environment.


In [ ]:
# 2. Automated Dataset Download
DATA_URL = "https://raw.githubusercontent.com/Ankit152/IMDB-sentiment-analysis/master/IMDB-Dataset.csv"
LOCAL_CSV = "IMDB-Dataset.csv"

if not os.path.exists(LOCAL_CSV):
    print(f"Downloading dataset from {DATA_URL}...")
    start = time.time()
    urllib.request.urlretrieve(DATA_URL, LOCAL_CSV)
    print(f"Downloaded {os.path.getsize(LOCAL_CSV) / (1024*1024):.2f} MB in {time.time() - start:.2f}s.")
else:
    print(f"Dataset already present locally ({os.path.getsize(LOCAL_CSV) / (1024*1024):.2f} MB).")

df = pd.read_csv(LOCAL_CSV)
print(f"Dataset Shape: {df.shape}")
df.head()


# 3. Schema, Missing Values & Duplicate Inspection
print("=== Dataset Info ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Duplicate Records ===")
dup_count = df.duplicated(subset=["review"]).sum()
print(f"Found {dup_count} duplicate reviews.")
if dup_count > 0:
    df = df.drop_duplicates(subset=["review"]).reset_index(drop=True)
    print(f"Unique dataset shape after deduplication: {df.shape}")

print("\n=== Class Distribution ===")
class_counts = df["sentiment"].value_counts()
print(class_counts)

# Visualize class distribution
plt.figure(figsize=(5, 3.5), dpi=100)
colors = ["#3b82f6", "#ef4444"]
plt.bar(class_counts.index, class_counts.values, color=colors, width=0.5)
plt.title("Target Class Distribution in IMDb Dataset", fontsize=11, fontweight="bold")
plt.xlabel("Sentiment Class")
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.show()


In [ ]:
# 3. Schema & Class Balance Inspection
print("=== Dataset Info ===")
df.info()

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Class Distribution ===")
class_counts = df['sentiment'].value_counts()
print(class_counts)

# Visualize class distribution
plt.figure(figsize=(5, 3.5), dpi=100)
colors = ['#3b82f6', '#ef4444']
plt.bar(class_counts.index, class_counts.values, color=colors, width=0.5)
plt.title('Target Class Distribution in IMDb Dataset', fontsize=11, fontweight='bold')
plt.xlabel('Sentiment Class')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.show()


---
## 4. Text Cleaning & Normalization
Web-scraped movie reviews frequently contain artifacts such as HTML tags (`<br /><br />`), HTML character entities (`&quot;`, `&amp;`), embedded URLs, and inconsistent spacing.
We implement a robust, production-grade text cleaner that:
1. Strips HTML tags using regular expressions.
2. Decodes HTML entities to canonical characters.
3. Removes URLs.
4. Normalizes whitespace and newlines.


In [ ]:
# 4. Text Cleaning Pipeline
HTML_TAG_PATTERN = re.compile(r"<[^>]+>")
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
WHITESPACE_PATTERN = re.compile(r"\s+")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Unescape HTML entities (&quot;, &#39;, &amp;)
    text = html.unescape(text)
    # Strip HTML tags
    text = HTML_TAG_PATTERN.sub(" ", text)
    # Strip URLs
    text = URL_PATTERN.sub(" ", text)
    # Collapse redundant whitespace
    text = WHITESPACE_PATTERN.sub(" ", text)
    return text.strip()

# Demonstrate cleaning on raw review with HTML artifacts
sample_raw = df['review'].iloc[0]
sample_clean = clean_text(sample_raw)

print("--- RAW SAMPLE REVIEW (First 250 chars) ---")
print(sample_raw[:250])
print("\n--- CLEANED SAMPLE REVIEW (First 250 chars) ---")
print(sample_clean[:250])

# Clean all reviews in the dataframe
t0 = time.time()
df['clean_review'] = df['review'].apply(clean_text)
print(f"\nCleaned {len(df):,d} reviews in {time.time() - t0:.2f} seconds.")


---
## 5. Preventing Data Leakage: Train / Validation / Test Strategy
### Crucial Methodology:
1. **Never fit feature extractors on the whole dataset**: Fitting TF-IDF on test or validation data leaks vocabulary frequencies and inverse document frequencies from unseen samples into the model.
2. **Stratified Splitting**:
   - **70% Training Set (35,000 samples)**: For feature extraction fitting and model parameter learning.
   - **15% Validation Set (7,500 samples)**: For model comparison and hyperparameter tuning.
   - **15% Test Set (7,500 samples)**: Untouched until final evaluation of the selected champion model.
3. **scikit-learn Pipeline**: Bundles `TfidfVectorizer` and the classifier so that transformations are applied strictly to the training fold during `fit()`, and only transformed during `predict()`.


In [ ]:
# 5. Stratified Partitioning (70% Train, 15% Val, 15% Test)
X = df['clean_review'].values
y = df['sentiment'].values

# Step 1: Hold out 15% pure test set
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, stratify=y
)

# Step 2: Split remaining 85% into Train (70% total) and Val (15% total)
val_ratio = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=val_ratio, random_state=RANDOM_SEED, stratify=y_train_val
)

print("Partition Summary:")
print(f"  Training set:   {len(X_train):,d} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Validation set: {len(X_val):,d} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"  Test set:       {len(X_test):,d} samples ({len(X_test)/len(X)*100:.1f}%)")


---
## 6. Feature Extraction & Pipeline Construction
We configure `TfidfVectorizer` with:
- **`max_features=15000`**: Retains the top 15,000 most informative n-grams, reducing memory footprint to <1MB while retaining strong predictive accuracy.
- **`ngram_range=(1, 2)`**: Captures both unigrams ("good", "awful") and bigrams ("not good", "must watch", "waste time") to capture negations and compound sentiment expressions.
- **`sublinear_tf=True`**: Applies sublinear term frequency scaling ($1 + \log(tf)$) to prevent high-frequency words from dominating.
- **`stop_words='english'`**: Eliminates non-informative grammatical noise.


In [ ]:
# 6. Pipeline Construction
def create_sentiment_pipeline(classifier):
    tfidf = TfidfVectorizer(
        max_features=15000,
        ngram_range=(1, 2),
        sublinear_tf=True,
        strip_accents='unicode',
        lowercase=True,
        stop_words='english'
    )
    return Pipeline([
        ('tfidf', tfidf),
        ('classifier', classifier)
    ])

print("Pipeline factory ready.")


---
## 7. Model 1: Multinomial Naive Bayes
Naive Bayes provides a strong, fast probabilistic baseline for bag-of-words and n-gram text classification.


In [ ]:
# 7. Train & Evaluate Multinomial Naive Bayes
mnb_pipeline = create_sentiment_pipeline(MultinomialNB(alpha=1.0))

print("Fitting Multinomial Naive Bayes on training set (35,000 samples)...")
t_start = time.time()
mnb_pipeline.fit(X_train, y_train)
mnb_train_time = time.time() - t_start
print(f"Fitted MNB in {mnb_train_time:.2f} seconds.")

# Validation predictions
mnb_val_preds = mnb_pipeline.predict(X_val)
mnb_val_acc = accuracy_score(y_val, mnb_val_preds)
mnb_val_f1 = f1_score(y_val, mnb_val_preds, pos_label='positive')

print(f"MNB Validation Accuracy: {mnb_val_acc * 100:.2f}%")
print(f"MNB Validation F1-Score: {mnb_val_f1 * 100:.2f}%")


---
## 8. Model 2: Logistic Regression (L2 Regularized)
Logistic Regression finds an optimal linear decision boundary across the high-dimensional TF-IDF space with L2 weight regularization.


In [ ]:
# 8. Train & Evaluate Logistic Regression
lr_pipeline = create_sentiment_pipeline(
    LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=RANDOM_SEED)
)

print("Fitting Logistic Regression on training set (35,000 samples)...")
t_start = time.time()
lr_pipeline.fit(X_train, y_train)
lr_train_time = time.time() - t_start
print(f"Fitted Logistic Regression in {lr_train_time:.2f} seconds.")

# Validation predictions
lr_val_preds = lr_pipeline.predict(X_val)
lr_val_acc = accuracy_score(y_val, lr_val_preds)
lr_val_f1 = f1_score(y_val, lr_val_preds, pos_label='positive')

print(f"Logistic Regression Validation Accuracy: {lr_val_acc * 100:.2f}%")
print(f"Logistic Regression Validation F1-Score: {lr_val_f1 * 100:.2f}%")


---
## 9. Validation Model Comparison & Selection
We compare the validation performance of both architectures to select our champion deployment pipeline.


In [ ]:
# 9. Comparison Summary Table
comparison_df = pd.DataFrame({
    'Model': ['Multinomial Naive Bayes', 'Logistic Regression'],
    'Training Time (s)': [round(mnb_train_time, 2), round(lr_train_time, 2)],
    'Validation Accuracy (%)': [round(mnb_val_acc * 100, 2), round(lr_val_acc * 100, 2)],
    'Validation F1-Score (%)': [round(mnb_val_f1 * 100, 2), round(lr_val_f1 * 100, 2)],
})

print(comparison_df.to_string(index=False))

# Select Champion
champion_pipeline = lr_pipeline if lr_val_f1 >= mnb_val_f1 else mnb_pipeline
champion_name = "Logistic Regression" if lr_val_f1 >= mnb_val_f1 else "Multinomial Naive Bayes"
print(f"\n🏆 Champion Model Selected: {champion_name}")


---
## 10. Final Evaluation on Held-Out Test Set
We now evaluate the champion pipeline on the **held-out test set (7,500 reviews)** that was isolated prior to feature extraction and training.


In [ ]:
# 10. Test Set Evaluation
y_test_pred = champion_pipeline.predict(X_test)
y_test_probs = champion_pipeline.predict_proba(X_test)[:, 1] # Positive class probability

test_acc = accuracy_score(y_test, y_test_pred)
test_prec = precision_score(y_test, y_test_pred, pos_label='positive')
test_rec = recall_score(y_test, y_test_pred, pos_label='positive')
test_f1 = f1_score(y_test, y_test_pred, pos_label='positive')
y_test_binary = (y_test == 'positive').astype(int)
test_roc_auc = roc_auc_score(y_test_binary, y_test_probs)

print("=" * 60)
print("HELD-OUT TEST SET EVALUATION RESULTS")
print("=" * 60)
print(f"Accuracy:  {test_acc * 100:.2f}%")
print(f"Precision: {test_prec * 100:.2f}%")
print(f"Recall:    {test_rec * 100:.2f}%")
print(f"F1-Score:  {test_f1 * 100:.2f}%")
print(f"ROC-AUC:   {test_roc_auc:.4f}")
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_test_pred, digits=4))

# Confusion Matrix Heatmap
cm = confusion_matrix(y_test, y_test_pred, labels=['negative', 'positive'])

fig, ax = plt.subplots(figsize=(6, 5), dpi=120)
cax = ax.imshow(cm, cmap='Blues', interpolation='nearest')
fig.colorbar(cax)

labels = ['Negative', 'Positive']
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(labels, fontsize=11, fontweight='medium')
ax.set_yticklabels(labels, fontsize=11, fontweight='medium')
ax.set_xlabel('Predicted Sentiment', fontsize=12, labelpad=10)
ax.set_ylabel('True Sentiment', fontsize=12, labelpad=10)
ax.set_title('Test Set Confusion Matrix', fontsize=13, fontweight='bold', pad=12)

total = np.sum(cm)
for i in range(2):
    for j in range(2):
        val = cm[i, j]
        pct = (val / total) * 100
        color = 'white' if val > cm.max() / 2 else 'black'
        ax.text(j, i, f"{val:,d}\n({pct:.1f}%)", ha='center', va='center', color=color, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 11. Qualitative Error Analysis
Inspecting samples where the model misclassified predictions exposes fundamental linguistic failure modes in bag-of-words / n-gram models:
1. **Sarcasm and Irony**: Reviews using superficially glowing adjectives to ridicule a production.
2. **Mixed Reviews**: Reviews praising actors/cinematography while ultimately giving a negative score.
3. **Negation Traps**: Distant or complex negative syntactic constructions.


In [ ]:
# 11. Inspecting False Positives & False Negatives
test_indices = np.arange(len(y_test))

false_positives = test_indices[(y_test == 'negative') & (y_test_pred == 'positive')]
false_negatives = test_indices[(y_test == 'positive') & (y_test_pred == 'negative')]

print(f"Total False Positives (Negative labeled as Positive): {len(false_positives):,d}")
print(f"Total False Negatives (Positive labeled as Negative): {len(false_negatives):,d}")

print("\n--- Sample False Positive (Negative review predicted as Positive) ---")
fp_idx = false_positives[0]
print(f"Review Snippet: {X_test[fp_idx][:300]}...")
print(f"Predicted Prob Positive: {y_test_probs[fp_idx]:.4f}")

print("\n--- Sample False Negative (Positive review predicted as Negative) ---")
fn_idx = false_negatives[0]
print(f"Review Snippet: {X_test[fn_idx][:300]}...")
print(f"Predicted Prob Positive: {y_test_probs[fn_idx]:.4f}")


---
## 12. Model Export & Security Notice
### Critical Serialization Safety Note:
Python `joblib` and `pickle` serialization formats can execute arbitrary code upon deserialization.
- **Rule**: Only load model files created by trusted internal pipelines.
- **Production Practice**: Verify cryptographic hashes (SHA-256) of model artifacts before loading in production services.


In [ ]:
# 12. Model Serialization
os.makedirs('models', exist_ok=True)
MODEL_SAVE_PATH = 'models/sentiment_pipeline.joblib'

print(f"Exporting fitted pipeline to {MODEL_SAVE_PATH}...")
joblib.dump(champion_pipeline, MODEL_SAVE_PATH, compress=3)

file_size_mb = os.path.getsize(MODEL_SAVE_PATH) / (1024 * 1024)
print(f"Saved pipeline successfully! Model size: {file_size_mb:.2f} MB.")

# Sanity check: Reload model and test inference
loaded_pipeline = joblib.load(MODEL_SAVE_PATH)
sample_test = ["An unforgettable, brilliant film with tremendous heart!"]
prediction = loaded_pipeline.predict(sample_test)[0]
confidence = loaded_pipeline.predict_proba(sample_test)[0].max()

print(f"Verification Test: '{sample_test[0]}'")
print(f"Result: {prediction.upper()} (Confidence: {confidence * 100:.1f}%)")


---
## 13. Colab File Export Helper
If running inside Google Colab, execute the cell below to download the generated `.joblib` file directly to your local workstation for use in the Streamlit application.


In [ ]:
# 13. Download to Local Machine (Colab Only)
try:
    from google.colab import files
    files.download(MODEL_SAVE_PATH)
    print("Triggered browser download for sentiment_pipeline.joblib.")
except ImportError:
    print(f"Running in local environment; model artifact is at {os.path.abspath(MODEL_SAVE_PATH)}.")
